Google released [Gemma 4](https://blog.google/innovation-and-ai/technology/developers-tools/gemma-4/) yesterday — open-weight multimodal models under Apache 2.0. In a [companion post](2026-04-03-gemma4-vs-gemini.ipynb), we compared Gemma 4 and Gemini via the API. Here, we run Gemma 4 **entirely locally** on a Mac Studio using [mlx-vlm](https://github.com/Blaizzy/mlx-vlm) and measure:

1. **Qualitative examples** — scene understanding, chart reading, segmentation
2. **Quantitative benchmarks** — VQA accuracy on VQAv2 subset, object detection IoU on COCO samples
3. **Runtime performance** — tokens/sec, prompt processing speed, peak memory

**Hardware**: Mac Studio

## Setup

```bash
uv pip install -U mlx-vlm==0.4.4 mlx-lm mlx datasets pycocotools supervision
```

In [ ]:
import time, json, re, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import supervision as sv
from IPython.display import display, Markdown
import mlx.core as mx

%config InlineBackend.figure_format = 'retina'

print(f"MLX version: {mx.__version__}")
print(f"Metal device: {mx.default_device()}")

## Loading the Model

We test **Gemma 4 31B IT (4-bit)** via mlx-vlm. At 4-bit quantization it needs ~19 GB — well within 64 GB unified memory.

In [ ]:
from mlx_vlm import load, generate
from mlx_vlm.prompt_utils import apply_chat_template
from mlx_vlm.utils import load_config

MODEL_PATH = 'mlx-community/gemma-4-31b-it-4bit'

t0 = time.time()
model, processor = load(MODEL_PATH)
config = load_config(MODEL_PATH)
load_time = time.time() - t0
print(f"Model loaded in {load_time:.1f}s")
print(f"Model type: {config.get('model_type')}")

## Inference Helper

In [ ]:
def run_local(prompt_text, image_path=None, max_tokens=512, display_image=True, quiet=False):
    """Run Gemma 4 locally and return results with timing."""
    if image_path and display_image and not quiet:
        img = Image.open(image_path) if isinstance(image_path, str) else image_path
        fig, ax = plt.subplots(figsize=(4, 3))
        ax.imshow(img); ax.axis('off'); plt.tight_layout(); plt.show()

    # Save PIL images to temp file if needed
    if image_path and not isinstance(image_path, str):
        tmp = '/tmp/_mlx_tmp_img.png'
        image_path.save(tmp)
        image_path = tmp

    prompt = apply_chat_template(
        processor, config, prompt_text,
        num_images=1 if image_path else 0
    )

    t0 = time.time()
    result = generate(
        model, processor, prompt,
        image=image_path if image_path else None,
        max_tokens=max_tokens,
        verbose=False
    )
    elapsed = time.time() - t0

    out = {
        'response': result.text,
        'prompt_tokens': result.prompt_tokens,
        'gen_tokens': result.generation_tokens,
        'prompt_tps': result.prompt_tps,
        'gen_tps': result.generation_tps,
        'peak_mem_gb': result.peak_memory / (1024**3) if hasattr(result, 'peak_memory') and result.peak_memory else None,
        'time_s': elapsed,
    }

    if not quiet:
        display(Markdown(f"**Gemma 4 31B (4-bit, local)** | {elapsed:.1f}s | "
                         f"{out['gen_tokens']} tok | "
                         f"{out['gen_tps']:.1f} tok/s"))
        display(Markdown(out['response'][:2000]))
    return out

---

# Part 1: Qualitative Examples

## Scene Understanding

In [ ]:
results_scene = run_local(
    "Describe this image in 2-3 sentences. What is happening and what objects do you see?",
    image_path='classroom.jpg',
    max_tokens=256
)

## Chart Analysis

In [ ]:
np.random.seed(42)
epochs = np.arange(1, 21)
train_loss = 2.5 * np.exp(-0.15 * epochs) + 0.1 + np.random.normal(0, 0.05, 20)
val_loss = 2.5 * np.exp(-0.12 * epochs) + 0.3 + np.random.normal(0, 0.08, 20)
val_loss[14:] += np.linspace(0, 0.4, 6)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(epochs, train_loss, 'b-o', label='Train', ms=4)
ax1.plot(epochs, val_loss, 'r-s', label='Val', ms=4)
ax1.set(xlabel='Epoch', ylabel='Loss', title='Training vs Validation Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

bars = ['CNN', 'ResNet', 'ViT', 'Ours']
acc = [78.2, 85.6, 89.1, 92.3]
ax2.bar(bars, acc, color=['#aaa','#aaa','#aaa','#e74c3c'])
ax2.set(ylabel='Accuracy (%)', title='Model Comparison', ylim=(70,100))
for i, v in enumerate(acc): ax2.text(i, v+0.5, f'{v}%', ha='center', fontsize=9)
plt.tight_layout()
fig.savefig('/tmp/research_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
results_chart = run_local(
    """Analyze these two plots concisely:
1. Left: what does it show and at which epoch does overfitting begin?
2. Right: which model is best and by how much over the runner-up?""",
    image_path='/tmp/research_plot.png',
    max_tokens=300,
    display_image=False
)

## Segmentation

In [ ]:
results_seg = run_local(
    """Segment the main animal in this image.
Return JSON: {"polygon": [[x1,y1], [x2,y2], ...], "label": "..."}
Coordinates in [0, 1000] range. Use 20+ points.
Return ONLY valid JSON.""",
    image_path='happy-doggy.jpg',
    max_tokens=2048
)

In [ ]:
from matplotlib.patches import Polygon as MPLPoly

seg_img = Image.open('happy-doggy.jpg')
img_np = np.array(seg_img)
h, w = img_np.shape[:2]

fig, ax = plt.subplots(figsize=(6, 5))
ax.set_title('Gemma 4 31B (local) — Segmentation')
ax.imshow(img_np)

try:
    raw = results_seg['response'].strip()
    raw = re.sub(r'```json\s*', '', raw)
    raw = re.sub(r'```', '', raw)
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    seg = json.loads(m.group() if m else raw)
    pts_raw = seg.get('polygon') or seg.get('segmentation') or seg.get('points', [])
    pts = [(x * w / 1000, y * h / 1000) for x, y in pts_raw]
    ax.add_patch(MPLPoly(pts, closed=True, fill=True,
                         fc='red', alpha=0.35, ec='red', lw=2))
    ax.set_xlabel(f"{len(pts)} points — {seg.get('label', '?')}")
except Exception as e:
    ax.set_xlabel(f"Parse error: {e}")

ax.axis('off')
plt.tight_layout(); plt.show()

---

# Part 2: Quantitative Benchmarks

We evaluate on small subsets of standard datasets to get quantitative metrics.

## VQA: VQAv2 Subset

We sample 50 questions from [vqav2-small](https://huggingface.co/datasets/merve/vqav2-small) validation split and measure exact-match accuracy against the ground truth answer.

In [ ]:
from datasets import load_dataset

vqa_ds = load_dataset('merve/vqav2-small', split='validation', streaming=True)

# Sample 50 examples deterministically
np.random.seed(42)
N_VQA = 50

vqa_samples = []
for i, ex in enumerate(vqa_ds):
    if len(vqa_samples) >= N_VQA:
        break
    if np.random.random() < 0.05:  # ~5% sampling rate
        vqa_samples.append(ex)

print(f"Loaded {len(vqa_samples)} VQA samples")
print(f"Example: Q='{vqa_samples[0]['question']}', A='{vqa_samples[0]['multiple_choice_answer']}'")

In [ ]:
def vqa_exact_match(prediction, ground_truth):
    """Exact match accuracy (case-insensitive, strip punctuation)."""
    pred = prediction.strip().lower().rstrip('.').strip()
    gt = ground_truth.strip().lower().rstrip('.').strip()
    return 1.0 if pred == gt else 0.0

vqa_results = []
for i, sample in enumerate(vqa_samples):
    img = sample['image']
    question = sample['question']
    gt_answer = sample['multiple_choice_answer']

    # Save image temporarily
    tmp_path = '/tmp/_vqa_tmp.png'
    img.save(tmp_path)

    prompt = f"""Answer this question about the image with a single short answer (1-3 words max).
Question: {question}
Answer:"""

    r = run_local(prompt, image_path=tmp_path, max_tokens=20, quiet=True)
    pred = r['response'].strip().split('\n')[0]

    acc = vqa_exact_match(pred, gt_answer)
    vqa_results.append({
        'question': question,
        'prediction': pred,
        'gt_answer': gt_answer,
        'accuracy': acc,
        'time_s': r['time_s'],
        'gen_tps': r['gen_tps'],
    })

    if (i+1) % 10 == 0:
        running_acc = np.mean([r['accuracy'] for r in vqa_results])
        print(f"  [{i+1}/{N_VQA}] Running accuracy: {running_acc:.1%}")

overall_vqa_acc = np.mean([r['accuracy'] for r in vqa_results])
avg_time = np.mean([r['time_s'] for r in vqa_results])
avg_tps = np.mean([r['gen_tps'] for r in vqa_results])
print(f"\nVQA Accuracy: {overall_vqa_acc:.1%} ({len(vqa_results)} samples)")
print(f"Avg time per question: {avg_time:.1f}s")
print(f"Avg generation speed: {avg_tps:.1f} tok/s")

In [ ]:
# Show some correct and incorrect examples
correct = [r for r in vqa_results if r['accuracy'] > 0]
wrong = [r for r in vqa_results if r['accuracy'] == 0]

display(Markdown(f"### VQA Results: {overall_vqa_acc:.1%} accuracy on {len(vqa_results)} samples"))
display(Markdown(f"{len(correct)} correct, {len(wrong)} incorrect"))

display(Markdown("#### Sample correct predictions"))
for r in correct[:5]:
    print(f"  Q: {r['question']}")
    print(f"  Pred: {r['prediction']} | GT: {r['gt_answer']}")
    print()

display(Markdown("#### Sample incorrect predictions"))
for r in wrong[:5]:
    print(f"  Q: {r['question']}")
    print(f"  Pred: {r['prediction']} | GT: {r['gt_answer']}")
    print()

## Object Detection: COCO Subset

We take 20 COCO validation images and prompt Gemma 4 to return bounding boxes, then match predictions against ground truth.
All visualizations use [supervision](https://github.com/roboflow/supervision) — green/red boxes on the images show what the model got right and wrong.

In [ ]:
COCO_CATS = {
    1:'person',2:'bicycle',3:'car',4:'motorcycle',5:'airplane',6:'bus',7:'train',8:'truck',
    9:'boat',10:'traffic light',11:'fire hydrant',13:'stop sign',14:'parking meter',15:'bench',
    16:'bird',17:'cat',18:'dog',19:'horse',20:'sheep',21:'cow',22:'elephant',23:'bear',24:'zebra',
    25:'giraffe',27:'backpack',28:'umbrella',31:'handbag',32:'tie',33:'suitcase',34:'frisbee',
    35:'skis',36:'snowboard',37:'sports ball',38:'kite',39:'baseball bat',40:'baseball glove',
    41:'skateboard',42:'surfboard',43:'tennis racket',44:'bottle',46:'wine glass',47:'cup',
    48:'fork',49:'knife',50:'spoon',51:'bowl',52:'banana',53:'apple',54:'sandwich',55:'orange',
    56:'broccoli',57:'carrot',58:'hot dog',59:'pizza',60:'donut',61:'cake',62:'chair',63:'couch',
    64:'potted plant',65:'bed',67:'dining table',70:'toilet',72:'tv',73:'laptop',74:'mouse',
    75:'remote',76:'keyboard',77:'cell phone',78:'microwave',79:'oven',80:'toaster',81:'sink',
    82:'refrigerator',84:'book',85:'clock',86:'vase',87:'scissors',88:'teddy bear',
    89:'hair drier',90:'toothbrush',
}

# Aliases: common VLM outputs → canonical COCO names
COCO_NAME_ALIASES = {
    'tv monitor': 'tv', 'tvmonitor': 'tv', 'television': 'tv', 'monitor': 'tv',
    'mobile phone': 'cell phone', 'cellphone': 'cell phone', 'phone': 'cell phone',
    'smartphone': 'cell phone', 'iphone': 'cell phone',
    'sofa': 'couch', 'loveseat': 'couch',
    'diningtable': 'dining table', 'table': 'dining table', 'desk': 'dining table',
    'pottedplant': 'potted plant', 'houseplant': 'potted plant', 'plant': 'potted plant',
    'aeroplane': 'airplane', 'plane': 'airplane', 'jet': 'airplane',
    'motorbike': 'motorcycle', 'scooter': 'motorcycle',
    'couch': 'couch', 'wine': 'wine glass', 'glass': 'wine glass',
    'laptop computer': 'laptop', 'notebook': 'laptop',
    'bicycle': 'bicycle', 'bike': 'bicycle',
    'teddy': 'teddy bear', 'stuffed bear': 'teddy bear', 'plush bear': 'teddy bear',
    'traffic signal': 'traffic light', 'signal': 'traffic light',
    'automobile': 'car', 'vehicle': 'car',
    'human': 'person', 'man': 'person', 'woman': 'person', 'child': 'person',
    'boy': 'person', 'girl': 'person', 'people': 'person', 'pedestrian': 'person',
}


def normalize_label(label):
    if label is None:
        return None
    label = re.sub(r'[_-]+', ' ', str(label).strip().lower())
    label = re.sub(r'\s+', ' ', label)
    return COCO_NAME_ALIASES.get(label, label)


COCO_NAME_TO_ID = {normalize_label(name): cid for cid, name in COCO_CATS.items()}

# Minimum GT box area (pixels²) — VLMs can't see tiny objects
MIN_GT_AREA = 32 * 32

coco_ds = load_dataset('detection-datasets/coco', split='val', streaming=True)

N_OD = 20

od_samples = []
for ex in coco_ds:
    if len(od_samples) >= N_OD:
        break
    cats = [COCO_CATS.get(c) for c in ex['objects']['category'] if c in COCO_CATS]
    if len(cats) >= 2:
        od_samples.append(ex)

print(f'Loaded {len(od_samples)} COCO images')


In [ ]:
def coco_gt_objects(sample):
    """Extract GT objects, filtering tiny boxes that VLMs can't see."""
    objects = []
    for bbox, class_id in zip(sample['objects']['bbox'], sample['objects']['category']):
        label = COCO_CATS.get(class_id)
        if label is None:
            continue
        x1, y1, x2, y2 = [float(v) for v in bbox]
        if x2 <= x1 or y2 <= y1:
            continue
        area = (x2 - x1) * (y2 - y1)
        if area < MIN_GT_AREA:
            continue
        objects.append({'label': label, 'class_id': class_id, 'xyxy': [x1, y1, x2, y2]})
    return objects


def parse_gemma_detections(text, width, height):
    """Parse Gemma's JSON response into detection objects."""
    if not text:
        return []
    cleaned = re.sub(r'```(?:json)?\s*', '', text.strip())
    cleaned = re.sub(r'```', '', cleaned)
    match = re.search(r'\[[\s\S]*\]', cleaned)
    try:
        dets = json.loads(match.group(0) if match else cleaned)
    except Exception:
        return []
    if not isinstance(dets, list):
        return []
    objects = []
    for d in dets:
        if not isinstance(d, dict):
            continue
        label = normalize_label(d.get('label') or d.get('class') or d.get('name') or d.get('object'))
        box = d.get('box_2d') or d.get('box') or d.get('bbox') or d.get('bounding_box')
        if label not in COCO_NAME_TO_ID or not isinstance(box, (list, tuple)) or len(box) != 4:
            continue
        try:
            y1_n, x1_n, y2_n, x2_n = [float(v) for v in box]
        except (ValueError, TypeError):
            continue
        # Convert from 0-1000 normalized to pixel coordinates
        x1 = float(np.clip(x1_n / 1000 * width, 0, width))
        x2 = float(np.clip(x2_n / 1000 * width, 0, width))
        y1 = float(np.clip(y1_n / 1000 * height, 0, height))
        y2 = float(np.clip(y2_n / 1000 * height, 0, height))
        if x2 <= x1 + 1 or y2 <= y1 + 1:
            continue
        objects.append({
            'label': label, 'class_id': COCO_NAME_TO_ID[label],
            'xyxy': [x1, y1, x2, y2],
        })
    return objects


def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0


def match_predictions(preds, gts, iou_threshold=0.5):
    """Greedy matching: each pred matches its best GT (same class, IoU >= threshold).
    Returns per-prediction match info and overall metrics."""
    matched_gt = set()
    tp = 0
    pred_info = []  # per-prediction: matched?, iou, label
    for pred in preds:
        best_iou, best_j = 0.0, None
        for j, gt in enumerate(gts):
            if j in matched_gt or pred['class_id'] != gt['class_id']:
                continue
            iou = compute_iou(pred['xyxy'], gt['xyxy'])
            if iou > best_iou:
                best_iou, best_j = iou, j
        is_tp = best_j is not None and best_iou >= iou_threshold
        if is_tp:
            matched_gt.add(best_j)
            tp += 1
        pred_info.append({'tp': is_tp, 'iou': best_iou, 'label': pred['label']})

    n_pred, n_gt = len(preds), len(gts)
    precision = tp / n_pred if n_pred else 0.0
    recall = tp / n_gt if n_gt else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    # Mean IoU of TRUE POSITIVES only (not dragged down by FPs)
    tp_ious = [p['iou'] for p in pred_info if p['tp']]
    mean_tp_iou = float(np.mean(tp_ious)) if tp_ious else 0.0

    return {
        'tp': tp, 'n_pred': n_pred, 'n_gt': n_gt,
        'precision': precision, 'recall': recall, 'f1': f1,
        'mean_tp_iou': mean_tp_iou,
        'pred_info': pred_info,
        'matched_gt_indices': matched_gt,
    }


In [ ]:
def to_sv_detections(objects):
    """Convert object list to supervision Detections."""
    if not objects:
        return sv.Detections.empty()
    label_to_id = {l: i for i, l in enumerate(sorted({o['label'] for o in objects}))}
    return sv.Detections(
        xyxy=np.array([o['xyxy'] for o in objects], dtype=np.float32),
        class_id=np.array([label_to_id[o['label']] for o in objects], dtype=int),
        data={'class_name': [o['label'] for o in objects]},
    )


def annotate_image(image, objects):
    """Draw boxes + labels using supervision."""
    scene = np.array(image.convert('RGB')).copy()
    if not objects:
        return scene
    dets = to_sv_detections(objects)
    labels = [o['label'] for o in objects]
    box_ann = sv.BoxAnnotator(thickness=2, color_lookup=sv.ColorLookup.CLASS)
    lbl_ann = sv.LabelAnnotator(text_scale=0.4, text_padding=3, color_lookup=sv.ColorLookup.CLASS)
    scene = box_ann.annotate(scene=scene, detections=dets)
    scene = lbl_ann.annotate(scene=scene, detections=dets, labels=labels)
    return scene


def annotate_tp_fp(image, pred_objects, pred_info):
    """Draw predictions color-coded: green=TP, red=FP. Pure supervision, no cv2."""
    scene = np.array(image.convert('RGB')).copy()
    if not pred_objects:
        return scene

    tp_mask = np.array([p['tp'] for p in pred_info])

    # Split into TP and FP groups and annotate separately with fixed colors
    for is_tp, color in [(True, sv.Color(0, 200, 0)), (False, sv.Color(220, 50, 50))]:
        mask = tp_mask if is_tp else ~tp_mask
        indices = np.where(mask)[0]
        if len(indices) == 0:
            continue
        subset = [pred_objects[i] for i in indices]
        subset_info = [pred_info[i] for i in indices]
        xyxy = np.array([o['xyxy'] for o in subset], dtype=np.float32)
        dets = sv.Detections(xyxy=xyxy)
        tag = 'TP' if is_tp else 'FP'
        labels = [f"{o['label']} ({tag})" for o in subset]
        box_ann = sv.BoxAnnotator(thickness=2, color=color)
        lbl_ann = sv.LabelAnnotator(text_scale=0.4, text_padding=3, color=color)
        scene = box_ann.annotate(scene=scene, detections=dets)
        scene = lbl_ann.annotate(scene=scene, detections=dets, labels=labels)

    return scene


In [ ]:
od_prompt = """Detect ALL visible objects in this image and return bounding boxes.
Return ONLY a JSON array — no explanation, no markdown.
Each element: {"label": "<coco_class>", "box_2d": [y_min, x_min, y_max, x_max]}
Coordinates are integers in [0, 1000] (normalized to image size).
Use standard COCO class names: person, car, dog, chair, cup, etc.
Include ALL visible instances, even partially occluded ones."""

od_results = []
for i, sample in enumerate(od_samples):
    img = sample['image']
    w, h = img.size
    gt = coco_gt_objects(sample)
    tmp = '/tmp/_od_tmp.jpg'
    img.save(tmp)
    out = run_local(od_prompt, image_path=tmp, max_tokens=2048, quiet=True)
    preds = parse_gemma_detections(out['response'], w, h)
    metrics = match_predictions(preds, gt)
    metrics['image'] = img
    metrics['pred_objects'] = preds
    metrics['gt_objects'] = gt
    metrics['time_s'] = out['time_s']
    metrics['raw_response'] = out['response']
    od_results.append(metrics)
    if (i + 1) % 5 == 0:
        p = np.mean([r['precision'] for r in od_results])
        r = np.mean([r['recall'] for r in od_results])
        print(f'  [{i+1}/{N_OD}] P={p:.2f}  R={r:.2f}')

mean_p = np.mean([r['precision'] for r in od_results])
mean_r = np.mean([r['recall'] for r in od_results])
mean_f1 = np.mean([r['f1'] for r in od_results])
mean_iou = np.mean([r['mean_tp_iou'] for r in od_results])
total_tp = sum(r['tp'] for r in od_results)
total_pred = sum(r['n_pred'] for r in od_results)
total_gt = sum(r['n_gt'] for r in od_results)

print(f'\n--- COCO Detection ({N_OD} images, GT boxes ≥ {MIN_GT_AREA}px²) ---')
print(f'Precision:      {mean_p:.2f}  ({total_tp}/{total_pred} preds matched)')
print(f'Recall:         {mean_r:.2f}  ({total_tp}/{total_gt} GT matched)')
print(f'F1:             {mean_f1:.2f}')
print(f'Mean TP IoU:    {mean_iou:.2f}  (only matched predictions)')
print(f'Avg time/image: {np.mean([r["time_s"] for r in od_results]):.1f}s')


### GT vs. Predictions: Side-by-Side

In [ ]:
viz_idx = np.linspace(0, len(od_results) - 1, 6, dtype=int)

fig, axes = plt.subplots(len(viz_idx), 2, figsize=(14, 4 * len(viz_idx)))

for row, idx in enumerate(viz_idx):
    r = od_results[idx]
    gt_scene = annotate_image(r['image'], r['gt_objects'])
    pred_scene = annotate_tp_fp(r['image'], r['pred_objects'], r['pred_info'])

    axes[row, 0].imshow(gt_scene)
    axes[row, 0].set_title(f'Ground truth ({r["n_gt"]} objects)', fontsize=10)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(pred_scene)
    axes[row, 1].set_title(
        f'Gemma 4 — {r["tp"]} TP (green) + {r["n_pred"] - r["tp"]} FP (red)  |  '
        f'P={r["precision"]:.2f}  R={r["recall"]:.2f}',
        fontsize=10,
    )
    axes[row, 1].axis('off')

fig.suptitle('COCO Detection: supervision overlays (green=TP, red=FP)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### Detection Quality Breakdown

In [ ]:
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1 — Per-image P vs R scatter
precs = [r['precision'] for r in od_results]
recs  = [r['recall'] for r in od_results]
axes[0].scatter(recs, precs, s=60, alpha=0.7, edgecolors='black', linewidth=0.5, c='#06b6d4')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_xlim(-0.05, 1.05); axes[0].set_ylim(-0.05, 1.05)
axes[0].set_title('Per-image Precision vs Recall')
axes[0].axhline(mean_p, color='gray', ls='--', alpha=0.5)
axes[0].axvline(mean_r, color='gray', ls='--', alpha=0.5)
axes[0].text(mean_r + 0.02, mean_p + 0.04, f'mean ({mean_p:.2f}, {mean_r:.2f})',
             fontsize=8, color='gray')

# 2 — IoU distribution of all TPs
tp_ious = [p['iou'] for r in od_results for p in r['pred_info'] if p['tp']]
if tp_ious:
    axes[1].hist(tp_ious, bins=15, color='#22c55e', edgecolor='white', alpha=0.85)
    axes[1].axvline(0.5, color='red', ls='--', lw=2, label='IoU=0.5 threshold')
    axes[1].set_xlabel('IoU'); axes[1].set_ylabel('Count')
    axes[1].legend(fontsize=8)
axes[1].set_title(f'IoU of matched predictions (n={len(tp_ious)})')

# 3 — Per-class recall (top 10 GT classes)
gt_counts = Counter()
tp_counts = Counter()
for r in od_results:
    for obj in r['gt_objects']:
        gt_counts[obj['label']] += 1
    for j in r['matched_gt_indices']:
        tp_counts[r['gt_objects'][j]['label']] += 1

top = gt_counts.most_common(10)
cls_names = [c for c, _ in top]
cls_recall = [tp_counts[c] / gt_counts[c] for c in cls_names]
bar_colors = ['#22c55e' if v >= 0.5 else '#f97316' if v >= 0.25 else '#ef4444' for v in cls_recall]

axes[2].barh(cls_names[::-1], cls_recall[::-1], color=bar_colors[::-1])
axes[2].set_xlabel('Recall')
axes[2].set_xlim(0, 1.05)
axes[2].set_title('Per-class recall (top 10 by GT count)')
# Annotate counts
for k, (name, rec) in enumerate(zip(cls_names[::-1], cls_recall[::-1])):
    n = gt_counts[name]
    axes[2].text(rec + 0.02, k, f'{tp_counts[name]}/{n}', va='center', fontsize=8)

plt.suptitle('Detection Quality Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### Best and Worst Detections

In [ ]:
f1s = [r['f1'] for r in od_results]
best_i, worst_i = int(np.argmax(f1s)), int(np.argmin(f1s))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row, (idx, label) in enumerate([(best_i, 'Best'), (worst_i, 'Worst')]):
    r = od_results[idx]
    axes[row, 0].imshow(annotate_image(r['image'], r['gt_objects']))
    axes[row, 0].set_title(f'{label} (F1={r["f1"]:.2f}) — GT: {r["n_gt"]} objects', fontsize=11)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(annotate_tp_fp(r['image'], r['pred_objects'], r['pred_info']))
    axes[row, 1].set_title(
        f'{label} — Pred: {r["n_pred"]}  |  P={r["precision"]:.2f}  R={r["recall"]:.2f}',
        fontsize=11,
    )
    axes[row, 1].axis('off')

plt.suptitle('Best vs Worst detection (by F1)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Diagnose worst case
rw = od_results[worst_i]
gt_labels = Counter(o['label'] for o in rw['gt_objects'])
pred_labels = Counter(o['label'] for o in rw['pred_objects'])
print(f'Worst image — GT classes:   {dict(gt_labels)}')
print(f'Worst image — Pred classes: {dict(pred_labels)}')
missed = set(gt_labels) - set(pred_labels)
if missed:
    print(f'Missed entirely: {missed}')


---

# Part 3: Runtime Performance

## Prompt Length vs Throughput

How does generation speed change as prompt complexity increases? We test text-only and image prompts of varying length.

In [ ]:
perf_tests = [
    ('Short text', 'What is 2+2?', None, 50),
    ('Medium text', 'Explain the transformer architecture in detail, covering attention, positional encoding, and training.', None, 300),
    ('Long text', 'Write a detailed tutorial on building a CNN for image classification in PyTorch. Cover data loading, model architecture with conv/pool/fc layers, training loop, evaluation, and common pitfalls. Include code snippets.' , None, 500),
    ('Image + short', 'What is this?', 'classroom.jpg', 50),
    ('Image + medium', 'Describe everything you see in this image in detail.', 'classroom.jpg', 300),
    ('Image + long', 'Analyze this image. Describe the setting, count all people, describe their activities, identify all objects, estimate the time of day, and suggest what event is taking place. Be thorough.', 'classroom.jpg', 500),
]

perf_rows = []
for name, prompt, img, max_tok in perf_tests:
    print(f"Running: {name}...", end=' ')
    r = run_local(prompt, image_path=img, max_tokens=max_tok, quiet=True)
    perf_rows.append({
        'Test': name,
        'Prompt tokens': r['prompt_tokens'],
        'Gen tokens': r['gen_tokens'],
        'Prompt tok/s': round(r['prompt_tps'], 1),
        'Gen tok/s': round(r['gen_tps'], 1),
        'Time (s)': round(r['time_s'], 1),
        'Peak mem (GB)': round(r['peak_mem_gb'], 1) if r['peak_mem_gb'] else 'N/A',
    })
    print(f"{r['time_s']:.1f}s, {r['gen_tps']:.1f} tok/s")

df_perf = pd.DataFrame(perf_rows)
display(df_perf.style.hide(axis='index'))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

labels = df_perf['Test']
colors = ['#3498db' if 'Image' not in t else '#e74c3c' for t in labels]

axes[0].barh(labels, df_perf['Gen tok/s'], color=colors, alpha=0.8)
axes[0].set_xlabel('Tokens/sec'); axes[0].set_title('Generation Speed')
for i, v in enumerate(df_perf['Gen tok/s']):
    axes[0].text(v + 0.3, i, f'{v}', va='center', fontsize=9)

axes[1].barh(labels, df_perf['Prompt tok/s'], color=colors, alpha=0.8)
axes[1].set_xlabel('Tokens/sec'); axes[1].set_title('Prompt Processing Speed')
for i, v in enumerate(df_perf['Prompt tok/s']):
    axes[1].text(v + 0.3, i, f'{v}', va='center', fontsize=9)

axes[2].barh(labels, df_perf['Time (s)'], color=colors, alpha=0.8)
axes[2].set_xlabel('Seconds'); axes[2].set_title('Total Time')
for i, v in enumerate(df_perf['Time (s)']):
    axes[2].text(v + 0.3, i, f'{v}s', va='center', fontsize=9)

fig.suptitle('Gemma 4 31B (4-bit) on M2 Max — Blue: text only, Red: with image', fontsize=11)
plt.tight_layout(); plt.show()

---

# Summary

In [ ]:
summary = pd.DataFrame([
    {"Metric": "Model", "Value": "Gemma 4 31B IT (4-bit)"},
    {"Metric": "Framework", "Value": "mlx-vlm 0.4.4 (PyPI)"},
    {"Metric": "Hardware", "Value": "Mac Studio M2 Max, 64 GB"},
    {"Metric": "Peak Memory", "Value": "~19 GB"},
    {"Metric": "Gen Speed (text)", "Value": f"{df_perf.iloc[0]['Gen tok/s']} tok/s"},
    {"Metric": "Gen Speed (vision)", "Value": f"{df_perf.iloc[3]['Gen tok/s']} tok/s"},
    {"Metric": "VQA Accuracy (VQAv2, n=50)", "Value": f"{overall_vqa_acc:.1%}"},
    {
        "Metric": "Localization Precision (COCO, n=20)",
        "Value": f"{np.mean([r['precision'] for r in od_results]):.2f}",
    },
    {
        "Metric": "Localization Recall (COCO, n=20)",
        "Value": f"{np.mean([r['recall'] for r in od_results]):.2f}",
    },
    {
        "Metric": "Localization Mean IoU (COCO, n=20)",
        "Value": f"{np.mean([r['avg_iou'] for r in od_results]):.2f}",
    },
])
display(summary.style.hide(axis="index"))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(["VQA Accuracy"], [overall_vqa_acc], color="#2ecc71", alpha=0.8)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Accuracy")
axes[0].set_title(f"VQAv2 ({len(vqa_results)} samples)")
axes[0].text(0, overall_vqa_acc + 0.02, f"{overall_vqa_acc:.1%}", ha="center", fontsize=14)

od_bar = {
    "Precision": np.mean([r["precision"] for r in od_results]),
    "Recall": np.mean([r["recall"] for r in od_results]),
    "Mean TP IoU": np.mean([r["mean_tp_iou"] for r in od_results]),
}
axes[1].bar(od_bar.keys(), od_bar.values(), color=["#3498db", "#e74c3c", "#f39c12"], alpha=0.8)
axes[1].set_ylim(0, 1)
axes[1].set_title(f"COCO Class-Aware Localization ({N_OD} images)")
for j, (label, value) in enumerate(od_bar.items()):
    axes[1].text(j, value + 0.02, f"{value:.2f}", ha="center", fontsize=12)

plt.suptitle("Gemma 4 31B (4-bit, local) — Quantitative Results", fontsize=12)
plt.tight_layout()
plt.show()


## Takeaways

- **Gemma 4 31B (4-bit) runs well locally** on M2 Max 64 GB with workable latency for interactive image prompting
- **mlx-vlm 0.4.4** on PyPI includes the box output path used here, and `supervision` makes the qualitative inspection much clearer
- **VQA performance** is solid on the VQAv2 subset, though the model can still be verbose when the answer should be a single token
- **COCO localization** is good enough for rough grounded prompting, but class-aware COCO matching still shows the gap between a general VLM and a dedicated detector
- **Class labels matter**: the notebook now scores only matches that overlap and agree on the predicted class, which is a more honest view than box overlap alone
- **Privacy** remains the main win: the full pipeline stays offline on Apple Silicon

For API-based comparison with Gemini models, see the [companion post](2026-04-03-gemma4-vs-gemini.ipynb).
